# 03 Gold Tables (B): Products, sellers, calendar, reviews

**Owner:** Person B
**Builds:** `gold.dim_geography`, `gold.dim_products`, `gold.dim_sellers`, `gold.dim_date`, `gold.fact_reviews`, `gold.mart_seller_performance`, `gold.mart_daily_sales_weather`
**Rule:** joins happen only in gold, never in silver. Aggregate one-to-many tables to one row per key BEFORE joining.

## Build order
1. `dim_geography`
2. `dim_products`, `dim_sellers`
3. `dim_date`
4. `fact_reviews`
5. `mart_seller_performance` and `mart_daily_sales_weather` (need Person A's `fact_orders`, `fact_order_items`, `dim_customers`)

---

## 1. `gold.dim_geography`
- **Grain:** one row per `zip_code_prefix`.
- **Base:** `silver.geolocation`
- **Joins:** none (silver is already averaged to one row per prefix).
- **Columns:** `zip_code_prefix`, `lat`, `lng`, `city`, `state`.
- **Checks:** `zip_code_prefix` unique; row count equals `silver.geolocation`.

## 2. `gold.dim_products`
- **Grain:** one row per `product_id`.
- **Base:** `silver.products`
- **Joins:**

| Join | Type | Columns produced |
|---|---|---|
| `silver.category_translation` on `category_name` | `LEFT JOIN` | `category_name_english` |

- **Columns:** `product_id`, `category_name`, `category_name_english`, `name_length`, `description_length`, `photos_qty`, `weight_g`, `length_cm`, `height_cm`, `width_cm`.
- **Fallback:** if there is no translation, `category_name_english` = `category_name`; if both are NULL, use `'unknown'`.
- **Checks:** `product_id` unique; row count equals `silver.products` (a higher count means the translation table has duplicates); count of untranslated categories is reported.

## 3. `gold.dim_sellers`
- **Grain:** one row per `seller_id`.
- **Base:** `silver.sellers`
- **Joins:** none. `zip_code_prefix` stays as a key to `dim_geography`.
- **Columns:** `seller_id`, `zip_code_prefix`, `city`, `state`.
- **Checks:** `seller_id` unique; row count equals `silver.sellers`.

## 4. `gold.dim_date`
- **Grain:** one row per calendar day (2016-01-01 to 2018-12-31, covering the order dates).
- **Base:** generated calendar (recursive CTE or numbers table).
- **Calendar columns:** `date`, `year`, `quarter`, `month`, `month_name`, `week`, `day_of_week`, `is_weekend`.
- **Joins (all `LEFT JOIN` on `date`):**

| Join | Columns produced | Note |
|---|---|---|
| `silver.holidays` | `is_holiday`, `holiday_name` | `is_holiday` = 0 when there is no match |
| `silver.exchange_rates` | `usd_per_brl` | Fill gaps (weekends) with the last known rate; confirm the rule before coding |
| `silver.weather` | `sp_temp_max`, `sp_temp_min`, `sp_precipitation` | **São Paulo only.** The `sp_` prefix keeps this visible downstream |

- **Checks:** no missing days; row count equals the number of days in the range; each joined table adds no duplicate dates; weather columns are NULL outside the weather date range, not 0.
- **Power BI:** mark as the date table.

## 5. `gold.fact_reviews`
- **Grain:** one row per `order_id`.
- **Base:** `silver.reviews`
- **Joins:** none. Aggregate by `order_id` instead:
  - `AVG(review_score)` → `review_score_avg`
  - `MIN(review_score)` → `review_score_min`
  - `COUNT(*)` → `review_count`
  - `MIN(review_creation_date)` → `first_review_date`
- **Columns:** `order_id`, `review_score_avg`, `review_score_min`, `review_count`, `first_review_date`.
- **Checks:** `order_id` unique; every score is between 1 and 5; every `order_id` exists in `gold.fact_orders` (run after A finishes).

## 6. `gold.mart_seller_performance`
- **Grain:** one row per `seller_id`.
- **Base:** `gold.fact_order_items`
- **Joins:**

| Join | Type | Aggregate first | Purpose |
|---|---|---|---|
| `gold.fact_order_items` | base | one row per `seller_id` + `order_id` (`SUM(price)`, `COUNT(*)`) | seller revenue and items per order |
| `gold.fact_orders` on `order_id` | `LEFT JOIN` | none (already one row per order) | `delivery_days`, `is_late` |
| `gold.fact_reviews` on `order_id` | `LEFT JOIN` | none (already one row per order) | `review_score_avg` |
| `gold.dim_sellers` on `seller_id` | `LEFT JOIN` | none | `city`, `state` |

- **Columns:** `seller_id`, `city`, `state`, `orders`, `items_sold`, `revenue`, `avg_review_score`, `late_pct`, `avg_delivery_days`, `has_enough_orders` (1 if `orders` >= 10).
- **Logic:**
  - Aggregate items to one row per `seller_id` + `order_id` before joining to orders and reviews.
  - Count `late_pct` and `avg_delivery_days` only on delivered orders.
  - An order with several sellers is counted once for each of them, so the review and lateness are shared across those sellers. Note this in the log.
- **Why a mart:** the per-seller rollup mixes three fact tables, which is clumsy in DAX.
- **Checks:** `seller_id` unique; `SUM(revenue)` equals `SUM(price)` in `fact_order_items`; `avg_review_score` is between 1 and 5.

## 7. `gold.mart_daily_sales_weather`
- **Grain:** one row per day, **São Paulo customers only** (`state = 'SP'`).
- **Base:** `gold.dim_date`
- **Joins:**

| Join | Type | Aggregate first | Purpose |
|---|---|---|---|
| `gold.fact_orders` + `gold.dim_customers` (filter `state = 'SP'`) by `order_date` | `LEFT JOIN` | `COUNT(*)` orders per day | `orders` |
| `gold.fact_order_items` on `order_id` | `LEFT JOIN` | `SUM(price)` per `order_id`, then per day | `revenue` |
| `gold.fact_orders` + `gold.dim_customers` (SP) by `delivered_customer_date` | `LEFT JOIN` | `COUNT(*)` and `SUM(is_late)` per delivered day | `delivered_orders`, `late_orders` |
| `silver.weather` (already in `dim_date` as `sp_*`) | none | none | `sp_precipitation`, `sp_temp_max`, `sp_temp_min` |

- **Columns:** `date`, `orders`, `revenue`, `delivered_orders`, `late_orders`, `late_pct`, `sp_precipitation`, `sp_temp_max`, `sp_temp_min`, `is_rainy` (precipitation above the chosen threshold), `is_holiday`, `is_weekend`.
- **Logic:**
  - Orders and revenue are grouped by the **purchase date**; late deliveries are grouped by the **delivered date**, because weather matters when the package is on the road.
  - Aggregate each source to one row per day in its own subquery, then join them to `dim_date`, so the sources do not multiply each other.
  - Days without orders get 0, not NULL. Days without weather stay NULL.
- **Limitation:** weather covers São Paulo only. Mark every visual that uses this mart "SP only".
- **Checks:** `date` unique; `SUM(orders)` equals the number of SP orders in `fact_orders`; `SUM(revenue)` equals SP item revenue.

## Validation (run at the end)
- Row counts: gold vs silver for every table above.
- No duplicate keys.
- Orphan checks: every `fact_order_items.product_id` exists in `dim_products` and every `seller_id` exists in `dim_sellers`.